# Catalogues STAC et exécution par blocs

**Cartomize · Version 1.0**

Durée indicative : 30 minutes.

## Objectifs

Lire un catalogue, télécharger ses assets, exploiter les métadonnées et comparer deux configurations de calcul.

## Apport de Cartomize

Le téléchargement enregistre les empreintes et le manifeste des scènes ; les paramètres d’exécution restent indépendants des formules scientifiques.

Données : paysage pédagogique généré avec une graine fixe. Les cartes et mesures ne décrivent aucun site réel.

## Préparation

Installer l’environnement décrit dans le [guide de démarrage](../README.md), puis exécuter les cellules dans l’ordre. Chaque exécution utilise un nouveau dossier de résultats.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.py').is_file() or (p/'tutorials/course.py').is_file())
root = root if (root/'course.py').is_file() else root/'tutorials'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import cartomize as cm
from course import demo, run_directory, show_raster, show_rgb, gallery, CRS, CLASSES, ROLES

In [ ]:
out = run_directory("10_catalogues_execution")
d = demo(out/"data")

## 1. Utiliser un catalogue local de démonstration

Le petit serveur ci-dessous sert des bandes simulées sur l’ordinateur. Il exerce les mêmes fonctions HTTP de recherche et téléchargement qu’un catalogue distant, sans présenter ses réponses comme des observations satellitaires. Le service est fermé à la fin du bloc.

In [ ]:
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler
from functools import partial
from threading import Thread
from contextlib import contextmanager
import hashlib

@contextmanager
def local_catalog(image):
    class Handler(SimpleHTTPRequestHandler):
        def log_message(self,*args): pass
        def do_GET(self):
            if not self.path.startswith('/search?'):
                return super().do_GET()
            body=json.dumps(collection).encode()
            self.send_response(200);self.send_header('Content-Type','application/json');self.end_headers();self.wfile.write(body)
    server=ThreadingHTTPServer(('127.0.0.1',0),partial(Handler,directory=str(image.parent)))
    base=f'http://127.0.0.1:{server.server_port}'
    with rasterio.open(image) as src:
        from rasterio.warp import transform_bounds
        bbox=list(transform_bounds(src.crs,'EPSG:4326',*src.bounds))
    item={'type':'Feature','stac_version':'1.0.0','id':'synthetic-course',
          'bbox':bbox,'geometry':{'type':'Polygon','coordinates':[[[bbox[0],bbox[1]],[bbox[2],bbox[1]],[bbox[2],bbox[3]],[bbox[0],bbox[3]],[bbox[0],bbox[1]]]]},
          'properties':{'datetime':'2026-01-01T00:00:00Z','platform':'synthetic'},'links':[],
          'assets':{'reflectance':{'href':base+'/'+image.name,'type':'image/tiff; application=geotiff',
              'eo:bands':[{'name':r,'common_name':{'swir1':'swir16','swir2':'swir22','rededge1':'rededge'}.get(r,r)} for r in ROLES],
              'raster:bands':[{'scale':1.,'offset':0.,'unit':'reflectance','nodata':'nan'} for _ in ROLES],
              'file:checksum':'1220'+hashlib.sha256(image.read_bytes()).hexdigest()}}}
    collection={'type':'FeatureCollection','features':[item],'links':[]}
    thread=Thread(target=server.serve_forever,daemon=True);thread.start()
    try: yield base
    finally: server.shutdown();server.server_close();thread.join()

with local_catalog(d['image']) as endpoint:
    found=cm.search_stac(endpoint,limit=1)
    search_path=out/'recherche.json';search_path.write_text(json.dumps(found))
    downloaded=cm.download_stac(search_path,out/'telechargement',assets=['reflectance'],max_bytes=5_000_000)
scenes=cm.discover_scenes(downloaded)
assert len(scenes)==1
assert set(scenes[0].bands)==(set(ROLES)-{'rededge1'})|{'rededge'}
local_item=json.loads(downloaded.read_text())['features'][0]
scene=cm.scene_from_stac(local_item,base=downloaded.parent)
assert scene.scene_id==scenes[0].scene_id
pd.DataFrame([{'scène':s.scene_id,'bandes':len(s.bands)} for s in scenes])

## 2. Vérifier les paramètres de calcul

La comparaison mesure le temps sur cet exemple. Elle ne démontre pas une accélération sur de grandes images : les coûts de démarrage peuvent dominer. Contrôler l’égalité des valeurs avant toute conclusion de performance.

In [ ]:
from time import perf_counter
rows=[];results=[]
for workers in (1,2):
    start=perf_counter()
    path=cm.spectral_indices(d['image'],out/f'ndvi_{workers}.tif',['NDVI'],workers=workers,block_size=64)
    rows.append({'workers':workers,'secondes':round(perf_counter()-start,4)})
    with rasterio.open(path) as src:results.append(src.read())
assert np.allclose(*results,equal_nan=True)
display(pd.DataFrame(rows))
display(cm.execution_capabilities())

## 3. Passer à un service distant

Remplacer `endpoint` par un service STAC Item Search et préciser `bbox`, `datetime_range` et `collections`. Vérifier la licence, la calibration et les assets retenus. Les services demandant une signature d’URL ou une authentification propre au fournisseur nécessitent une préparation supplémentaire. Dask et CUDA sont facultatifs ; leur présence ne garantit pas un gain de vitesse.

## Exercice

Limiter le téléchargement à 1 octet. Quel comportement protège le dossier de résultats ?

<details>
<summary>Éléments de correction</summary>

`download_stac` refuse le téléchargement qui dépasse `max_bytes` et ne publie pas de dossier partiel. Utiliser un nouveau dossier pour cet essai.

</details>

## Reproduction

Les paramètres, la graine et les chemins sont explicites dans les cellules. Le répertoire `out` contient les produits de cette exécution. Adapter les sources, le système de coordonnées et les paramètres avant de transférer la méthode à une étude.